# Notebook 05: ML Forecasting - Prophet Time Series

## Overview
This notebook trains Prophet models to forecast cholera cases 4 weeks ahead with uncertainty intervals.

## Prerequisites
- Notebook 04 completed successfully
- Gold tables: `epi_analytics_weekly`, `epi_country_trends`
- Prophet library installed (or mock_prophet for testing)

## Inputs
- Historical case data from Gold layer

## Outputs
- `gold.forecast_cases_4wk` - 4-week ahead forecasts with uncertainty
- `gold.model_performance` - Model evaluation metrics

## Execution Time
~2-3 minutes

In [ ]:
# ============================================
# ENVIRONMENT DETECTION & CONFIGURATION
# ============================================

import os
import sys
from pathlib import Path

IS_FABRIC = os.path.exists('/lakehouse/default')

if IS_FABRIC:
    print("🌐 Running in Microsoft Fabric")
    GOLD_TABLE_PATH = "/lakehouse/default/Tables/gold"
else:
    print("💻 Running locally")
    project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    sys.path.insert(0, str(project_root / 'src'))
    GOLD_TABLE_PATH = str(project_root / "data" / "gold_tables")

print(f"Gold Path: {GOLD_TABLE_PATH}")

In [ ]:
# ============================================
# IMPORTS
# ============================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import logging

# Try to import Prophet, fall back to mock if not available
try:
    from prophet import Prophet
    USE_REAL_PROPHET = True
    print("✅ Using real Prophet library")
except ImportError:
    try:
        from epi_analytics.mock_prophet import Prophet
        USE_REAL_PROPHET = False
        print("⚠️  Using mock Prophet (for testing without Stan backend)")
    except ImportError:
        print("❌ Neither Prophet nor mock_prophet available")
        raise

# Import forecasting functions
try:
    from epi_analytics.forecasting import train_prophet_model, explain_forecast
    print("✅ Imported forecasting functions")
except ImportError:
    print("⚠️  Forecasting module not available, using inline functions")
    train_prophet_model = None
    explain_forecast = None

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ Imports successful")

In [ ]:
# ============================================
# LOAD GOLD DATA
# ============================================

print("\n📂 Loading Gold layer data...\n")

try:
    if IS_FABRIC:
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()
        df_weekly = spark.table("gold.epi_analytics_weekly").toPandas()
        df_country_trends = spark.table("gold.epi_country_trends").toPandas()
    else:
        df_weekly = pd.read_parquet(Path(GOLD_TABLE_PATH) / "epi_analytics_weekly.parquet")
        df_country_trends = pd.read_parquet(Path(GOLD_TABLE_PATH) / "epi_country_trends.parquet")
    
    print(f"✅ Loaded {len(df_weekly)} weekly records")
    print(f"✅ Loaded {len(df_country_trends)} country trend records")
    
except Exception as e:
    logger.error(f"Error loading Gold data: {e}")
    raise

In [ ]:
# ============================================
# PREPARE FORECASTING DATA
# ============================================

print("\n🔄 Preparing forecasting dataset...\n")

# Use weekly aggregated data for continental forecast
df_forecast_input = df_weekly[['date', 'new_cases']].copy()

# Convert date to datetime
if df_forecast_input['date'].dtype == 'object':
    df_forecast_input['date'] = pd.to_datetime(df_forecast_input['date'])

# Rename columns for Prophet (requires 'ds' and 'y')
df_forecast_input = df_forecast_input.rename(columns={
    'date': 'ds',
    'new_cases': 'y'
})

# Sort by date
df_forecast_input = df_forecast_input.sort_values('ds').reset_index(drop=True)

print(f"✅ Prepared {len(df_forecast_input)} weeks of historical data")
print(f"   - Date range: {df_forecast_input['ds'].min()} to {df_forecast_input['ds'].max()}")
print(f"   - Total cases: {df_forecast_input['y'].sum():,}")
print(f"   - Average weekly cases: {df_forecast_input['y'].mean():.0f}")

# Display sample
print("\n📋 Sample Forecast Input:")
display(df_forecast_input.head())

In [ ]:
# ============================================
# TRAIN PROPHET MODEL
# ============================================

print("\n🤖 Training Prophet model...\n")

FORECAST_WEEKS = 4

if train_prophet_model:
    # Use imported function
    model, forecast, metrics = train_prophet_model(df_forecast_input, forecast_weeks=FORECAST_WEEKS)
else:
    # Inline training
    model = Prophet(
        yearly_seasonality=False,  # Not enough data for yearly
        weekly_seasonality=True,
        daily_seasonality=False,
        interval_width=0.95
    )
    
    model.fit(df_forecast_input)
    
    # Create future dataframe
    future = model.make_future_dataframe(periods=FORECAST_WEEKS, freq='W')
    forecast = model.predict(future)
    
    # Calculate metrics on historical data
    historical_forecast = forecast.iloc[:len(df_forecast_input)]
    y_true = df_forecast_input['y'].values
    y_pred = historical_forecast['yhat'].values
    
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    
    metrics = {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

print("✅ Model training complete")
print(f"\n📊 Model Performance Metrics:")
print(f"   - MAE (Mean Absolute Error): {metrics['MAE']:.2f}")
print(f"   - RMSE (Root Mean Squared Error): {metrics['RMSE']:.2f}")
print(f"   - MAPE (Mean Absolute Percentage Error): {metrics['MAPE']:.2f}%")

In [ ]:
# ============================================
# EXTRACT FORECAST RESULTS
# ============================================

print("\n📈 Extracting forecast results...\n")

# Get future forecast only (last 4 weeks)
df_forecast_future = forecast.tail(FORECAST_WEEKS).copy()

# Add metadata
df_forecast_future['forecast_date'] = datetime.now().date()
df_forecast_future['forecast_horizon_weeks'] = FORECAST_WEEKS
df_forecast_future['model_type'] = 'Prophet'
df_forecast_future['model_version'] = '1.0'

# Calculate confidence level
df_forecast_future['confidence_level'] = 95  # 95% prediction interval

# Rename columns for clarity
df_forecast_future = df_forecast_future.rename(columns={
    'ds': 'forecast_week_end_date',
    'yhat': 'predicted_cases',
    'yhat_lower': 'predicted_cases_lower',
    'yhat_upper': 'predicted_cases_upper'
})

# Round predictions
df_forecast_future['predicted_cases'] = df_forecast_future['predicted_cases'].round(0).astype(int)
df_forecast_future['predicted_cases_lower'] = df_forecast_future['predicted_cases_lower'].round(0).astype(int)
df_forecast_future['predicted_cases_upper'] = df_forecast_future['predicted_cases_upper'].round(0).astype(int)

# Ensure non-negative predictions
df_forecast_future['predicted_cases'] = df_forecast_future['predicted_cases'].clip(lower=0)
df_forecast_future['predicted_cases_lower'] = df_forecast_future['predicted_cases_lower'].clip(lower=0)
df_forecast_future['predicted_cases_upper'] = df_forecast_future['predicted_cases_upper'].clip(lower=0)

# Add audit column
df_forecast_future['created_at'] = datetime.now()

# Select columns
forecast_cols = [
    'forecast_week_end_date', 'predicted_cases', 
    'predicted_cases_lower', 'predicted_cases_upper',
    'confidence_level', 'forecast_date', 'forecast_horizon_weeks',
    'model_type', 'model_version', 'created_at'
]

df_forecast_future = df_forecast_future[forecast_cols]

print(f"✅ Extracted {len(df_forecast_future)} weeks of forecasts")
print("\n📋 4-Week Forecast:")
display(df_forecast_future[[
    'forecast_week_end_date', 'predicted_cases', 
    'predicted_cases_lower', 'predicted_cases_upper'
]])

In [ ]:
# ============================================
# MODEL EXPLAINABILITY
# ============================================

print("\n🔍 Extracting model explainability...\n")

if explain_forecast:
    explanation = explain_forecast(model, forecast)
else:
    # Inline explainability
    explanation = {
        'trend': 'Increasing' if forecast['trend'].iloc[-1] > forecast['trend'].iloc[0] else 'Decreasing',
        'seasonality': 'Weekly pattern detected' if USE_REAL_PROPHET else 'Mock seasonality',
        'forecast_weeks': len(df_forecast_future)
    }

print("📊 Model Explanation:")
print(f"   - Trend: {explanation.get('trend', 'N/A')}")
print(f"   - Seasonality: {explanation.get('seasonality', 'N/A')}")
print(f"   - Forecast horizon: {explanation.get('forecast_weeks', FORECAST_WEEKS)} weeks")

# Extract trend component if available
if 'trend' in forecast.columns:
    trend_change = forecast['trend'].iloc[-1] - forecast['trend'].iloc[0]
    print(f"   - Trend change: {trend_change:+.2f} cases")

In [ ]:
# ============================================
# CREATE MODEL PERFORMANCE TABLE
# ============================================

print("\n📊 Creating model performance record...\n")

df_model_performance = pd.DataFrame([{
    'model_id': 'prophet_continental_v1',
    'model_type': 'Prophet',
    'model_version': '1.0',
    'training_date': datetime.now().date(),
    'training_records': len(df_forecast_input),
    'forecast_horizon_weeks': FORECAST_WEEKS,
    'mae': metrics['MAE'],
    'rmse': metrics['RMSE'],
    'mape': metrics['MAPE'],
    'seasonality_mode': 'additive',
    'interval_width': 0.95,
    'created_at': datetime.now()
}])

print("✅ Model performance record created")
display(df_model_performance[[
    'model_id', 'training_records', 'mae', 'rmse', 'mape'
]])

In [ ]:
# ============================================
# SAVE TO GOLD LAYER
# ============================================

print("\n💾 Saving forecasts to Gold layer...\n")

try:
    if IS_FABRIC:
        spark_forecast = spark.createDataFrame(df_forecast_future)
        spark_performance = spark.createDataFrame(df_model_performance)
        
        spark_forecast.write.format("delta").mode("overwrite").saveAsTable("gold.forecast_cases_4wk")
        spark_performance.write.format("delta").mode("overwrite").saveAsTable("gold.model_performance")
        
        print("✅ Delta tables created in Fabric Lakehouse")
    else:
        # Convert datetime columns to strings
        for df, name in [
            (df_forecast_future, 'forecast_cases_4wk'),
            (df_model_performance, 'model_performance')
        ]:
            df_save = df.copy()
            for col in df_save.columns:
                if df_save[col].dtype == 'datetime64[ns]' or 'datetime' in str(df_save[col].dtype):
                    df_save[col] = df_save[col].astype(str)
            
            df_save.to_parquet(
                Path(GOLD_TABLE_PATH) / f"{name}.parquet",
                index=False,
                engine='pyarrow'
            )
            print(f"✅ Saved {name}.parquet ({len(df_save)} rows)")
        
        print(f"\n✅ All files saved to: {GOLD_TABLE_PATH}")
        
except Exception as e:
    logger.error(f"Error saving forecasts: {e}")
    raise

print("\n✅ ML forecasting complete!")

## Validation & Testing

In [ ]:
# ============================================
# VALIDATION & TESTING
# ============================================

print("\n🔍 Running validation checks...\n")

# Test 1: Forecast generated
assert len(df_forecast_future) == FORECAST_WEEKS, f"Expected {FORECAST_WEEKS} forecasts, got {len(df_forecast_future)}"
print(f"✅ Generated {FORECAST_WEEKS}-week forecast")

# Test 2: Predictions are non-negative
assert (df_forecast_future['predicted_cases'] >= 0).all(), "Negative predictions found"
assert (df_forecast_future['predicted_cases_lower'] >= 0).all(), "Negative lower bounds found"
print(f"✅ All predictions are non-negative")

# Test 3: Uncertainty intervals valid
assert (df_forecast_future['predicted_cases_upper'] >= df_forecast_future['predicted_cases']).all(), "Upper bound < prediction"
assert (df_forecast_future['predicted_cases'] >= df_forecast_future['predicted_cases_lower']).all(), "Prediction < lower bound"
print(f"✅ Uncertainty intervals valid")

# Test 4: Model performance metrics calculated
assert metrics['MAE'] > 0, "MAE not calculated"
assert metrics['RMSE'] > 0, "RMSE not calculated"
print(f"✅ Model performance metrics calculated")

# Test 5: Forecast summary
print("\n📊 Forecast Summary:")
print(f"   - Forecast period: {df_forecast_future['forecast_week_end_date'].min()} to {df_forecast_future['forecast_week_end_date'].max()}")
print(f"   - Total predicted cases (4 weeks): {df_forecast_future['predicted_cases'].sum():,}")
print(f"   - Average weekly prediction: {df_forecast_future['predicted_cases'].mean():.0f}")
print(f"   - Prediction range: {df_forecast_future['predicted_cases'].min():,} to {df_forecast_future['predicted_cases'].max():,}")
print(f"   - Average uncertainty width: {(df_forecast_future['predicted_cases_upper'] - df_forecast_future['predicted_cases_lower']).mean():.0f} cases")

print("\n✅ All validation checks passed!")

## Next Steps

1. **Review forecast results** above
2. **Monitor forecast accuracy** as actual data comes in
3. **Retrain model** weekly with new data
4. **Use forecasts** in Power BI dashboard

## Outputs Created

- `gold.forecast_cases_4wk` - 4-week ahead forecasts with 95% prediction intervals
- `gold.model_performance` - Model evaluation metrics and metadata

**Pipeline Complete!** 🎉 All 5 notebooks executed successfully.